In [28]:
NO_RECORDS = 102

In [29]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
import os
from fastapi import FastAPI, Query, HTTPException
import logging

from lxml import etree as ET
# API calls
import requests



from utils.enrichment_methods import get_doi, clean_doi, batch_enrich_subjects_from_openaire, merge_subjects_from_new_method, connect_ids_with_subjects_by_doi


from utils.enrichment_methods import batch_enrich_subjects_from_openalex, _openalex_batch_doi_resolve, batch_enrich_subjects_from_datacite

In [30]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [31]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [32]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        ARRAY(
            SELECT elem->>'subject'
            FROM jsonb_array_elements(datacite_json->'subjects') AS elem
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    doi,
    subjects
FROM query
WHERE 
array_length(subjects, 1) IS NULL AND
-- for these tests, we also want the records to have valid DOIs
     doi IS NOT NULL
ORDER BY id
LIMIT %s
;
"""

In [33]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results[0])

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_dois = [row["doi"] for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/102734481', 'title': 'Water structure around surfactant micelles', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/102734481</identifier>\n    <datestamp>2019-03-18T15:13:33.990Z</datestamp>\n    <setSpec>doiInvs</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier identifierType="DOI">https://doi.org/10.5286/ISIS.E.RB620144-1</identifier>\n 

# DOI check through bigger repositories  (#1)

## OpenAIRE

In [34]:
openaire_enriched = batch_enrich_subjects_from_openaire(record_dois)
openaire_enriched[:5]

DEBUG:    2026-07-03 11:32:29 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-07-03 11:32:30 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.5286%2Fisis.e.rb620144-1%2C10.5286%2Fisis.e.rb1790051-7%2C10.5286%2Fisis.e.rb1830641-1%2C10.5286%2Fisis.e.rb1830641-2%2C10.5286%2Fisis.e.rb1790346-2%2C10.5286%2Fisis.e.rb1790346-3%2C10.5286%2Fisis.e.rb1790346-5%2C10.5286%2Fisis.e.rb1790346-4%2C10.5286%2Fisis.e.rb1790346-6%2C10.5286%2Fisis.e.rb1790346-7 HTTP/1.1" 200 None
DEBUG:    2026-07-03 11:32:30 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-07-03 11:32:31 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.5286%2Fisis.e.rb1670547-3%2C10.5286%2Fisis.e.rb1990292-1%2C10.5286%2Fisis.e.rb1890354-2%2C10.5286%2Fisis.e.rb1510265-2%2C10.5286%2Fisis.e.rb1920617-3%2C10.5286%2Fisis.e.rb1790175-3%2C10.5286%2Fisis.e.rb1

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': '10.5286/isis.e.rb1790051-7',
  'enriched_subjects': {'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1790346-2',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}}]

In [35]:
print(len(record_dois))

102


In [36]:
succesfully_enriched = [el for el in openaire_enriched if el['enriched_subjects']]
print(len(succesfully_enriched))

24


In [37]:
print(len(succesfully_enriched) / len(record_dois) * 100)

23.52941176470588


In [38]:
initial_set = set([clean_doi(doi) for doi in record_dois])
enriched_set = set([clean_doi(el['doi']) for el in openaire_enriched])

In [39]:
initial_set - enriched_set

set()

In [40]:
enriched_set - initial_set

set()

If both sets are empty, no DOIs have been added during enrichment and all initial DOIs have been preserved even with no subjects.

In [41]:
# To create dictionaries for later joining of subjects:
id_doi_pairs = [{"id": id, "doi": doi}
    for (id, doi) 
    in zip(record_ids[:5], record_dois[:5])]

id_doi_pairs

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/102734481',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218557',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1790051-7'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218657',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-1'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218664',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-2'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224154',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-2'}]

In [42]:
connect_ids_with_subjects_by_doi(id_doi_pairs, openaire_enriched)

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/102734481',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218557',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1790051-7',
  'enriched_subjects': {'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218657',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103218664',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224154',
  'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-2',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
  

## OpenAlex API

In [43]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [44]:
test_dois = [
    '10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.34894/YGHDXB',
    '10.15151/esrf-es-1580558984',
    '10.15151/esrf-es-2323561778',
    '10.15151/ESRF-ES-1030397492'
]

In [45]:
batch_enrich_subjects_from_openalex(test_dois, OpenAlex_API_key)

DEBUG:    2026-07-03 11:32:39 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-03 11:32:39 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.1016/j.paid.2021.110888%7Chttps://doi.org/10.23662/fors-ds-686-1%7Chttps://doi.org/10.34894/yghdxb%7Chttps://doi.org/10.15151/esrf-es-1580558984%7Chttps://doi.org/10.15151/esrf-es-2323561778%7Chttps://doi.org/10.15151/esrf-es-1030397492&per_page=100&api_key=nFYjNvBt180q2JQXDpT62V HTTP/1.1" 200 None


[{'doi': 'https://doi.org/10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Anxiety',
   'Big Five personality traits',
   'Conscientiousness',
   'Context (archaeology)',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Personality',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'doi': '10.23662/fors-ds-686-1', 'enriched_subjects': set()},
 {'doi': '10.34894/YGHDXB', 'enriched_subjects': set()},
 {'doi': '10.15151/esrf-es-1580558984', 'enriched_subjects': set()},
 {'doi': '10.15151/esrf-es-2323561778', 'enriched_subjects': set()},
 {'doi': '10.15151/ESRF-ES-1030397492', 'enriched_subjects': set()}]

In [46]:
OA_0_0 = batch_enrich_subjects_from_openalex(record_dois,
                                    OpenAlex_API_key,
                                    score_treshold = 0.5)
OA_0_0

DEBUG:    2026-07-03 11:32:39 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-03 11:32:40 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.5286/isis.e.rb620144-1%7Chttps://doi.org/10.5286/isis.e.rb1790051-7%7Chttps://doi.org/10.5286/isis.e.rb1830641-1%7Chttps://doi.org/10.5286/isis.e.rb1830641-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-3%7Chttps://doi.org/10.5286/isis.e.rb1790346-5%7Chttps://doi.org/10.5286/isis.e.rb1790346-4%7Chttps://doi.org/10.5286/isis.e.rb1790346-6%7Chttps://doi.org/10.5286/isis.e.rb1790346-7%7Chttps://doi.org/10.5286/isis.e.rb1670547-3%7Chttps://doi.org/10.5286/isis.e.rb1990292-1%7Chttps://doi.org/10.5286/isis.e.rb1890354-2%7Chttps://doi.org/10.5286/isis.e.rb1510265-2%7Chttps://doi.org/10.5286/isis.e.rb1920617-3%7Chttps://doi.org/10.5286/isis.e.rb1790175-3%7Chttps://doi.org/10.5286/isis.e.rb1990395-3%7Chttps://doi.o

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790051-7',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-1',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-2',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-2',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-3',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-5',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-4',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-6',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-7',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1670547-3',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1990292-1',


In [47]:
len([el for el in OA_0_0 if el['enriched_subjects']])

0

In [48]:
OA_0_5 = batch_enrich_subjects_from_openalex(record_dois,
                                    OpenAlex_API_key,
                                    score_treshold = 0.5)
OA_0_5

DEBUG:    2026-07-03 11:32:40 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-03 11:32:41 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.5286/isis.e.rb620144-1%7Chttps://doi.org/10.5286/isis.e.rb1790051-7%7Chttps://doi.org/10.5286/isis.e.rb1830641-1%7Chttps://doi.org/10.5286/isis.e.rb1830641-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-3%7Chttps://doi.org/10.5286/isis.e.rb1790346-5%7Chttps://doi.org/10.5286/isis.e.rb1790346-4%7Chttps://doi.org/10.5286/isis.e.rb1790346-6%7Chttps://doi.org/10.5286/isis.e.rb1790346-7%7Chttps://doi.org/10.5286/isis.e.rb1670547-3%7Chttps://doi.org/10.5286/isis.e.rb1990292-1%7Chttps://doi.org/10.5286/isis.e.rb1890354-2%7Chttps://doi.org/10.5286/isis.e.rb1510265-2%7Chttps://doi.org/10.5286/isis.e.rb1920617-3%7Chttps://doi.org/10.5286/isis.e.rb1790175-3%7Chttps://doi.org/10.5286/isis.e.rb1990395-3%7Chttps://doi.o

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790051-7',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-1',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1830641-2',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-2',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-3',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-5',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-4',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-6',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1790346-7',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1670547-3',
  'enriched_subjects': set()},
 {'doi': 'https://doi.org/10.5286/ISIS.E.RB1990292-1',


In [49]:
len([el for el in OA_0_5 if el['enriched_subjects']])

0

In [50]:
OA_0_7 = batch_enrich_subjects_from_openalex(record_dois,
                                    OpenAlex_API_key,
                                    score_treshold = 0.7)

DEBUG:    2026-07-03 11:32:41 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openalex.org:443
DEBUG:    2026-07-03 11:32:42 urllib3.connectionpool@544 https://api.openalex.org:443 "GET /works?filter=doi:https://doi.org/10.5286/isis.e.rb620144-1%7Chttps://doi.org/10.5286/isis.e.rb1790051-7%7Chttps://doi.org/10.5286/isis.e.rb1830641-1%7Chttps://doi.org/10.5286/isis.e.rb1830641-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-2%7Chttps://doi.org/10.5286/isis.e.rb1790346-3%7Chttps://doi.org/10.5286/isis.e.rb1790346-5%7Chttps://doi.org/10.5286/isis.e.rb1790346-4%7Chttps://doi.org/10.5286/isis.e.rb1790346-6%7Chttps://doi.org/10.5286/isis.e.rb1790346-7%7Chttps://doi.org/10.5286/isis.e.rb1670547-3%7Chttps://doi.org/10.5286/isis.e.rb1990292-1%7Chttps://doi.org/10.5286/isis.e.rb1890354-2%7Chttps://doi.org/10.5286/isis.e.rb1510265-2%7Chttps://doi.org/10.5286/isis.e.rb1920617-3%7Chttps://doi.org/10.5286/isis.e.rb1790175-3%7Chttps://doi.org/10.5286/isis.e.rb1990395-3%7Chttps://doi.o

In [51]:
len([el for el in OA_0_7 if el['enriched_subjects']])

0

In [52]:
succesfully_enriched = [el for el in OA_0_7 if el['enriched_subjects']]
print(len(succesfully_enriched)/len(record_dois) * 100)

0.0


## DataCite API

In [53]:
datacite_enriched = batch_enrich_subjects_from_datacite(record_dois)
datacite_enriched[:5]

DEBUG:    2026-07-03 11:32:42 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-03 11:32:43 urllib3.connectionpool@544 https://api.datacite.org:443 "GET /dois/10.5286/isis.e.rb620144-1?disable-facets=true HTTP/1.1" 200 None
DEBUG:    2026-07-03 11:32:43 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-03 11:32:43 urllib3.connectionpool@544 https://api.datacite.org:443 "GET /dois/10.5286/isis.e.rb1790051-7?disable-facets=true HTTP/1.1" 200 None
DEBUG:    2026-07-03 11:32:43 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-03 11:32:43 urllib3.connectionpool@544 https://api.datacite.org:443 "GET /dois/10.5286/isis.e.rb1830641-1?disable-facets=true HTTP/1.1" 200 None
DEBUG:    2026-07-03 11:32:43 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.datacite.org:443
DEBUG:    2026-07-03 11:32:44 urllib3.connectionpool@544

[{'doi': 'https://doi.org/10.5286/ISIS.E.RB620144-1',
  'enriched_subjects': set()},
 {'doi': '10.5286/isis.e.rb1790051-7',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1830641-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'doi': '10.5286/isis.e.rb1790346-2',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}}]

In [54]:
succesfully_enriched = [el for el in datacite_enriched if el['enriched_subjects']]
print(len(succesfully_enriched)/len(record_dois) * 100)

24.509803921568626
